# 01 — Data Download

Downloads all raw data needed for the hot/cold × p16 analysis.

**Sources:**
- **UCSC Xena** — TCGA pan-cancer RNA-seq (log2(norm_count+1), gene-level), clinical data
- **NCI GDC PanImmune** — Thorsson et al. 2018 CIBERSORT immune cell fractions
- **cBioPortal** — CDKN2A copy-number and mutation calls (TCGA PanCancer Atlas)

All files are saved to `../data/`. That directory is gitignored — re-run this notebook to regenerate.

In [1]:
import os
import requests
from pathlib import Path
from tqdm.notebook import tqdm

DATA = Path("../data")
DATA.mkdir(exist_ok=True)

## 1. TCGA Pan-Cancer RNA-seq (UCSC Xena)

Gene expression: log2(norm_count + 1), 60,483 genes × ~11,000 TCGA samples.
File is ~1.3 GB compressed.

In [2]:
XENA_EXPR_URL = (
    "https://tcga-pancan-atlas-hub.s3.us-east-1.amazonaws.com/"
    "download/EB%2B%2BAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.xena.gz"
)
XENA_CLINICAL_URL = (
    "https://tcga-pancan-atlas-hub.s3.us-east-1.amazonaws.com/"
    "download/Survival_SupplementalTable_S1_20171025_xena_sp"
)
XENA_SAMPLE_TYPE_URL = (
    "https://tcga-pancan-atlas-hub.s3.us-east-1.amazonaws.com/"
    "download/TCGA_phenotype_denseDataOnlyDownload.tsv.gz"
)

def download_file(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    """Stream-download url to dest, skipping if already present."""
    if dest.exists():
        print(f"  Already downloaded: {dest.name}")
        return
    print(f"Downloading {dest.name} ...")
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
        for chunk in r.iter_content(chunk_size):
            f.write(chunk)
            bar.update(len(chunk))
    print(f"  Saved to {dest}")

download_file(XENA_EXPR_URL,         DATA / "tcga_pancan_expr.tsv.gz")
download_file(XENA_CLINICAL_URL,     DATA / "tcga_clinical_survival.tsv")
download_file(XENA_SAMPLE_TYPE_URL,  DATA / "tcga_sample_types.tsv.gz")

  Already downloaded: tcga_pancan_expr.tsv.gz
  Already downloaded: tcga_clinical_survival.tsv
  Already downloaded: tcga_sample_types.tsv.gz


## 2. Thorsson et al. 2018 — TCGA Immune Landscape (CIBERSORT Fractions)

Pre-computed immune cell fractions for all TCGA samples from CIBERSORT
(Thorsson et al. 2018, Cell / PMID 29628290). Hosted by NCI GDC PanImmune
project — downloaded programmatically, no manual step required.

In [3]:
import urllib.request, shutil
import pandas as pd

THORSSON_FILE = DATA / "thorsson2018_immune_landscape.tsv"
# NCI GDC PanImmune supplementary — TCGA.Kallisto.fullIDs.cibersort.relative.tsv
THORSSON_URL = "https://api.gdc.cancer.gov/data/b3df502e-3594-46ef-9f94-d041a20a0b9a"

if not THORSSON_FILE.exists():
    print(f"Downloading Thorsson 2018 immune landscape → {THORSSON_FILE}")
    with urllib.request.urlopen(THORSSON_URL) as r, open(THORSSON_FILE, "wb") as f:
        shutil.copyfileobj(r, f)
    print("Done.")
else:
    print(f"Already present: {THORSSON_FILE}")

# Sanity check
df = pd.read_csv(THORSSON_FILE, sep="\t", index_col=0, nrows=3)
print(f"Thorsson loaded: {df.shape} (preview)")
display(df.head(3))

Already present: ../data/thorsson2018_immune_landscape.tsv
Thorsson loaded: (3, 26) (preview)


,CancerType,B.cells.naive,B.cells.memory,Plasma.cells,T.cells.CD8,T.cells.CD4.naive,T.cells.CD4.memory.resting,T.cells.CD4.memory.activated,T.cells.follicular.helper,T.cells.regulatory..Tregs.,...,Macrophages.M2,Dendritic.cells.resting,Dendritic.cells.activated,Mast.cells.resting,Mast.cells.activated,Eosinophils,Neutrophils,P.value,Correlation,RMSE
SampleID,,,,,,,,,,,,,,,,,,,,,
TCGA.OR.A5JG.01A.11R.A29S.07,ACC,0.000000,0.048529,0.016052,0.046099,0.027037,0.290682,0,0.000000,0,...,0.363861,0.002715,0.026125,0.032788,0.00000,0.01029,0.009607,0.112,0.095797,1.047142
TCGA.OR.A5LG.01A.11R.A29S.07,ACC,0.007169,0.011125,0.007982,0.139842,0.000000,0.142742,0,0.001614,0,...,0.448243,0.000000,0.007464,0.126237,0.00000,0.00000,0.000000,0.104,0.103345,1.046163
TCGA.OR.A5JD.01A.11R.A29S.07,ACC,0.000023,0.014607,0.000000,0.104888,0.000000,0.174895,0,0.017928,0,...,0.329552,0.000000,0.009330,0.000000,0.19073,0.00000,0.000000,0.068,0.143259,1.039812


## 3. cBioPortal — CDKN2A Copy Number & Mutations (TCGA PanCancer Atlas)

We use the cBioPortal web API to pull CDKN2A data from the TCGA PanCancer Atlas study.

In [4]:
import pandas as pd
import json

CBIO_BASE = "https://www.cbioportal.org/api"
GENE = "CDKN2A"
CDKN2A_ENTREZ = 1029

def cbio_get(endpoint: str, params: dict | None = None) -> list:
    r = requests.get(f"{CBIO_BASE}/{endpoint}", params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def cbio_post(endpoint: str, body: dict, params: dict | None = None) -> list:
    r = requests.post(f"{CBIO_BASE}/{endpoint}", json=body, params=params, timeout=60)
    r.raise_for_status()
    return r.json() if r.content else []

# Get all TCGA PanCancer Atlas 2018 study IDs
all_studies = cbio_get("studies?keyword=pan_can_atlas&pageSize=50")
pan_can_study_ids = [s["studyId"] for s in all_studies if "pan_can_atlas_2018" in s["studyId"]]
print(f"Found {len(pan_can_study_ids)} PanCancer Atlas studies")

# -- Copy Number Alterations --
# Uses POST /molecular-profiles/{id}/discrete-copy-number/fetch
# sampleListId pattern: {study_id}_all
cna_file = DATA / "cdkn2a_cna.json"
if not cna_file.exists():
    print("Fetching CDKN2A CNA calls from cBioPortal ...")
    cna_data = []
    for study_id in pan_can_study_ids:
        profile_id = f"{study_id}_gistic"
        try:
            records = cbio_post(
                f"molecular-profiles/{profile_id}/discrete-copy-number/fetch",
                body={"entrezGeneIds": [CDKN2A_ENTREZ], "sampleListId": f"{study_id}_all"},
                params={"discreteCopyNumberEventType": "ALL"},
            )
            for rec in records:
                rec["studyId"] = study_id
            cna_data.extend(records)
        except Exception as e:
            print(f"  Skipping {study_id} CNA: {e}")
    cna_file.write_text(json.dumps(cna_data))
    print(f"  Saved {len(cna_data)} CNA records.")
else:
    print(f"Already downloaded: {cna_file.name}")

# -- Mutations --
# Uses POST /molecular-profiles/{id}/mutations/fetch
mut_file = DATA / "cdkn2a_mutations.json"
if not mut_file.exists():
    print("Fetching CDKN2A mutations from cBioPortal ...")
    mut_data = []
    for study_id in pan_can_study_ids:
        profile_id = f"{study_id}_mutations"
        try:
            records = cbio_post(
                f"molecular-profiles/{profile_id}/mutations/fetch",
                body={"entrezGeneIds": [CDKN2A_ENTREZ], "sampleListId": f"{study_id}_all"},
                params={"projection": "SUMMARY"},
            )
            for rec in records:
                rec["studyId"] = study_id
            mut_data.extend(records)
        except Exception as e:
            print(f"  Skipping {study_id} mutations: {e}")
    mut_file.write_text(json.dumps(mut_data))
    print(f"  Saved {len(mut_data)} mutation records.")
else:
    print(f"Already downloaded: {mut_file.name}")

Found 32 PanCancer Atlas studies
Fetching CDKN2A CNA calls from cBioPortal ...


  Saved 10712 CNA records.
Fetching CDKN2A mutations from cBioPortal ...


  Saved 422 mutation records.


## 3b. cBioPortal — Additional Senescence Loci (TP53, RB1, CDKN1A, CDKN2B)

In [ ]:
# Additional senescence-related genes: CNA and/or mutations from cBioPortal
# Aligned with SENESCENCE_GENOMIC_LOCI in src/signatures.py
# TP53 (7157), RB1 (5925), ATM (472), PTEN (5728), CDKN1A (1026), CDKN2B (1030), CDKN2C (1031)
EXTRA_GENES = [
    ("tp53",   7157, True,  True),   # (name, entrez, fetch_cna, fetch_mut)
    ("rb1",    5925, True,  True),
    ("atm",    472,  True,  True),   # DNA-damage sensor; deletion/mutation → senescence bypass
    ("pten",   5728, True,  True),   # PTEN loss → PI3K/AKT → CDK activation
    ("cdkn1a", 1026, True,  False),
    ("cdkn2b", 1030, True,  False),
    ("cdkn2c", 1031, True,  False),  # p18^INK4c — INK4 family member
]

for gene_name, entrez_id, fetch_cna, fetch_mut in EXTRA_GENES:
    # ── Copy Number ──────────────────────────────────────────────────────────
    if fetch_cna:
        cna_file = DATA / f"{gene_name}_cna.json"
        if not cna_file.exists():
            print(f"Fetching {gene_name.upper()} CNA ...")
            cna_data = []
            for study_id in pan_can_study_ids:
                profile_id = f"{study_id}_gistic"
                try:
                    records = cbio_post(
                        f"molecular-profiles/{profile_id}/discrete-copy-number/fetch",
                        body={"entrezGeneIds": [entrez_id], "sampleListId": f"{study_id}_all"},
                        params={"discreteCopyNumberEventType": "ALL"},
                    )
                    for rec in records:
                        rec["studyId"] = study_id
                    cna_data.extend(records)
                except Exception as e:
                    print(f"  Skipping {study_id} CNA: {e}")
            cna_file.write_text(json.dumps(cna_data))
            print(f"  Saved {len(cna_data)} CNA records → {cna_file.name}")
        else:
            print(f"Already downloaded: {gene_name}_cna.json")

    # ── Mutations ─────────────────────────────────────────────────────────────
    if fetch_mut:
        mut_file = DATA / f"{gene_name}_mutations.json"
        if not mut_file.exists():
            print(f"Fetching {gene_name.upper()} mutations ...")
            mut_data = []
            for study_id in pan_can_study_ids:
                profile_id = f"{study_id}_mutations"
                try:
                    records = cbio_post(
                        f"molecular-profiles/{profile_id}/mutations/fetch",
                        body={"entrezGeneIds": [entrez_id], "sampleListId": f"{study_id}_all"},
                        params={"projection": "SUMMARY"},
                    )
                    for rec in records:
                        rec["studyId"] = study_id
                    mut_data.extend(records)
                except Exception as e:
                    print(f"  Skipping {study_id} mutations: {e}")
            mut_file.write_text(json.dumps(mut_data))
            print(f"  Saved {len(mut_data)} mutation records → {mut_file.name}")
        else:
            print(f"Already downloaded: {gene_name}_mutations.json")

## 3c. MSigDB Gene Set Caching

Fetch and cache the MSigDB gene sets used for ssGSEA scoring (senescence, SASP, IFN-γ, inflammatory response) into a local JSON file so notebook 03 can load them without network access.

In [ ]:
import json as _json

MSIGDB_CACHE = DATA / "msigdb_genesets.json"
MSIGDB_SETS_TO_FETCH = [
    "FRIDMAN_SENESCENCE_UP",
    "REACTOME_CELLULAR_SENESCENCE",
    "SAUL_SEN_MAYO",
    "HALLMARK_P53_PATHWAY",
    "GOBP_CELLULAR_SENESCENCE",
    "FRIDMAN_SENESCENCE_DN",
    "HALLMARK_INTERFERON_GAMMA_RESPONSE",
    "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
]

if not MSIGDB_CACHE.exists():
    print("Caching MSigDB gene sets ...")
    import urllib.request as _ur
    cached = {}
    for set_name in MSIGDB_SETS_TO_FETCH:
        url = (
            f"https://www.gsea-msigdb.org/gsea/msigdb/download_geneset.jsp"
            f"?geneSetName={set_name}&fileType=txt"
        )
        try:
            req = _ur.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with _ur.urlopen(req, timeout=30) as r:
                lines = r.read().decode().strip().splitlines()
            genes = [l.strip() for l in lines[2:] if l.strip()]
            cached[set_name] = genes
            print(f"  {set_name}: {len(genes)} genes")
        except Exception as e:
            print(f"  Warning: could not fetch {set_name}: {e}")
    MSIGDB_CACHE.write_text(_json.dumps(cached, indent=2))
    print(f"Saved → {MSIGDB_CACHE}")
else:
    print(f"Already cached: {MSIGDB_CACHE.name}")
    cached = _json.loads(MSIGDB_CACHE.read_text())
    print(f"  Sets cached: {list(cached.keys())}")

## 3d. Tumor Purity (TCGA ABSOLUTE)

Consensus purity estimates from Aran et al. 2015 (*Nature Communications*), derived using the ABSOLUTE algorithm. Hosted on the UCSC Xena pan-cancer hub.

In [ ]:
# TCGA consensus purity estimates (Aran et al. 2015, Nat Comms)
# ABSOLUTE purity/ploidy from UCSC Xena pan-cancer hub
PURITY_URL = (
    "https://tcga-pancan-atlas-hub.s3.us-east-1.amazonaws.com/"
    "download/TCGA_mastercalls.abs_tables_JSedit.fixed.txt"
)
purity_file = DATA / "tcga_purity.tsv"
download_file(PURITY_URL, purity_file)

# Sanity check
if purity_file.exists():
    _pur = pd.read_csv(purity_file, sep="\t", nrows=3)
    print(f"Purity columns: {list(_pur.columns)}")

## 3e. Thorsson Immune Subtypes (C1–C6)

Pan-cancer immune subtype assignments from Thorsson et al. 2018. Six subtypes (C1 wound healing, C2 IFN-γ dominant, C3 inflammatory, C4 lymphocyte depleted, C5 immunologically quiet, C6 TGF-β dominant). Hosted on UCSC Xena pan-cancer hub.

In [ ]:
# Thorsson et al. 2018 immune subtype assignments (C1–C6)
# Hosted on UCSC Xena pan-cancer hub
IMMUNE_SUBTYPE_URL = (
    "https://tcga-pancan-atlas-hub.s3.us-east-1.amazonaws.com/"
    "download/Subtype_Immune_Model_Based.txt.gz"
)
immune_subtype_file = DATA / "tcga_immune_subtypes.tsv.gz"
download_file(IMMUNE_SUBTYPE_URL, immune_subtype_file)

if immune_subtype_file.exists():
    _ist = pd.read_csv(immune_subtype_file, sep="\t", compression="gzip", nrows=3)
    print(f"Immune subtype columns: {list(_ist.columns)}")

## 3f. Tumor Mutational Burden (TMB) from cBioPortal

Non-synonymous mutations per megabase fetched as a clinical attribute (`TMB_NONSYNONYMOUS`) from each TCGA PanCancer Atlas study via the cBioPortal REST API.

In [ ]:
# Tumor mutational burden — nonsynonymous mutations per Mb
# Fetched as a clinical attribute from each PanCancer Atlas study
tmb_file = DATA / "tcga_tmb.json"
if not tmb_file.exists():
    print("Fetching TMB from cBioPortal ...")
    tmb_records = []
    for study_id in pan_can_study_ids:
        try:
            records = cbio_get(
                f"studies/{study_id}/clinical-data",
                params={"clinicalDataType": "SAMPLE", "attributeId": "TMB_NONSYNONYMOUS"}
            )
            for rec in records:
                rec["studyId"] = study_id
            tmb_records.extend(records)
        except Exception as e:
            print(f"  Skipping {study_id} TMB: {e}")
    tmb_file.write_text(json.dumps(tmb_records))
    print(f"  Saved {len(tmb_records)} TMB records.")
else:
    print(f"Already downloaded: {tmb_file.name}")
    _tmb = json.loads(tmb_file.read_text())
    print(f"  {len(_tmb)} records")

## 4. Sanity Checks

In [5]:
import gzip

# Peek at expression matrix
expr_file = DATA / "tcga_pancan_expr.tsv.gz"
if expr_file.exists():
    with gzip.open(expr_file, "rt") as fh:
        header = fh.readline().strip().split("\t")
    print(f"Expression matrix: {len(header)-1} samples (columns)")

# Clinical table
clin_file = DATA / "tcga_clinical_survival.tsv"
if clin_file.exists():
    clin = pd.read_csv(clin_file, sep="\t", nrows=5)
    print(f"Clinical columns: {list(clin.columns)}")

print("\nData directory contents:")
for f in sorted(DATA.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:50s} {size_mb:7.1f} MB")

Expression matrix: 11069 samples (columns)
Clinical columns: ['sample', '_PATIENT', 'cancer type abbreviation', 'age_at_initial_pathologic_diagnosis', 'gender', 'race', 'ajcc_pathologic_tumor_stage', 'clinical_stage', 'histological_type', 'histological_grade', 'initial_pathologic_dx_year', 'menopause_status', 'birth_days_to', 'vital_status', 'tumor_status', 'last_contact_days_to', 'death_days_to', 'cause_of_death', 'new_tumor_event_type', 'new_tumor_event_site', 'new_tumor_event_site_other', 'new_tumor_event_dx_days_to', 'treatment_outcome_first_course', 'margin_status', 'residual_tumor', 'OS', 'OS.time', 'DSS', 'DSS.time', 'DFI', 'DFI.time', 'PFI', 'PFI.time', 'Redaction']

Data directory contents:
  cdkn2a_cna.json                                        3.9 MB
  cdkn2a_mutations.json                                  0.3 MB
  sample_immune_labels.parquet                           0.2 MB
  ssgsea_senescence_scores.parquet                       0.2 MB
  tcga_clinical_survival.tsv       